In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
sns.set_theme(style='whitegrid',context='notebook',palette='colorblind')
SELECTED = CONFIG['models']
ORDER = ['empirical_complete'] + SELECTED
PALETTE = dict(zip(ORDER,sns.color_palette('colorblind',len(ORDER))))
PALETTE['se_hurdle_s'] = '#333333'
DASHES = dict(zip(ORDER,['',(4,2),(1,1),(6,2,1,2),(3,1,1,1),(8,2),(2,1,2,3)]))
MARKERS = dict(zip(ORDER,['o','s','^','D','v','P','X']))
def save(fig,name):
    fig.tight_layout()
    fig.savefig(OUTPUT/(name+'.png'),dpi=180,bbox_inches='tight')
    fig.savefig(OUTPUT/(name+'.pdf'),bbox_inches='tight')
    plt.show()
    plt.close(fig)
def lines(data,x,y,ax):
    sns.lineplot(data=data,x=x,y=y,hue='source',style='source',hue_order=ORDER,style_order=ORDER,
                 palette=PALETTE,dashes=DASHES,markers=MARKERS,markersize=3,errorbar=None,ax=ax)
def legend(ax):
    ax.legend(fontsize=7,ncol=2)
def cdf_lines(data,x,y,ax):
    # duplicate x values draw vertical jumps; no averaging of left/right ECDF limits
    for source in ORDER:
        d = data.loc[data.source.eq(source)].sort_values(x,kind='stable')
        sns.lineplot(data=d,x=x,y=y,estimator=None,sort=False,color=PALETTE[source],
                     linestyle='-' if not DASHES[source] else (0,DASHES[source]),label=source,ax=ax)
    ax.set_ylim(-.02,1.02)

In [ ]:
counts = pd.read_csv(INPUT/'count_moments.csv')
deltas = pd.read_csv(INPUT/'increment_moments.csv')
METRICS = ['mean','median','variance','sd','skewness','excess_kurtosis']

In [ ]:
# full and GRW-excluded views; exclusion affects only the view
for kind,table in [('count',counts),('increment',deltas)]:
    for scale in ['raw','offset_log']:
        for suffix,excluded in [('',[]),('_without_grw',['lognormal_grw'])]:
            subset = table.loc[table.scale.eq(scale)&table.source.isin(ORDER)&~table.source.isin(excluded)]
            fig,axes = plt.subplots(2,3,figsize=(14,8),sharex=True)
            for ax,metric in zip(axes.flat,METRICS):
                lines(subset,'age',metric,ax)
                if scale=='raw' and not excluded:ax.set_yscale('symlog',linthresh=1)
                ax.axvline(5.5 if kind=='count' else 4.5,color='.6',ls=':',lw=1)
                ax.set(xlabel='Model year' if kind=='count' else 'Transition start year',ylabel=metric.replace('_',' '),title=metric.replace('_',' '))
                if ax!=axes.flat[0]:ax.legend().remove()
            legend(axes.flat[0])
            fig.suptitle(f'{kind.title()} moments — {scale}; AR(6,6) empirical prefix before dotted boundary')
            save(fig,f'{kind}_moments_{scale}{suffix}')